# DuckLake to MotherDuck: Validate locally, deploy to cloud in minutes


This Colab is an extension of the blog [DuckLake to MotherDuck: Validate locally, deploy to cloud in minutes
](https://dlthub.com/blog/ducklake-to-motherduck-with-dlt) . It shows how to build and test your pipeline locally, then switch to MotherDuck without changing your code. It keeps the workflow simple and helps you trust the handoff to the cloud.

> 📚 Docs: [dlt pipelines](https://dlthub.com/docs/general-usage/pipeline) | [DuckLake destination](https://dlthub.com/docs/dlt-ecosystem/destinations/ducklake) | [MotherDuck destination](https://dlthub.com/docs/dlt-ecosystem/destinations/motherduck)

## Install dependencies

We'll pin the notebook to the exact stack we ship in the demo so both DuckLake and MotherDuck extras are available.

In [ ]:
%%capture
%pip install -q "dlt[ducklake,motherduck, dbml, workspace]" duckdb==1.4.2

*Required duckdb version: 1.4.2


## Define the Hacker News source

We will fetch top stories directly from the Hacker News API using dlt's REST API source. The REST API source uses a parent resource (`top_story_ids`) to seed story IDs, and a dependent resource (`stories`) that resolves each ID to fetch the full story details. dlt resources yield clean Python dictionaries, so DuckLake can infer the schema automatically.

In [ ]:
import dlt
import requests

session = requests.Session()

# fetch top Hacker News stories
@dlt.resource(
    table_name="stories",
    write_disposition="merge",
    primary_key="id"
)
def hacker_news(limit=30):
    ids = session.get(
        "https://hacker-news.firebaseio.com/v0/topstories.json"
    ).json()[:limit]

    for story_id in ids:
        story = session.get(
            f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json"
        ).json()
        if story:
            yield story

pipeline = dlt.pipeline(
    pipeline_name="hn_local",
    destination="ducklake",
    dataset_name="hacker_news"
)

# run the pipeline with 'ducklake' destination
print(pipeline.run(hacker_news(50)))

Pipeline hn_local load step completed in 5.52 seconds
1 load package(s) were loaded to destination ducklake and into dataset hacker_news
The ducklake destination used ducklake@sqlite:////content/ducklake.sqlite@file:///content/ducklake.files location to store data
Load package 1765439232.0146315 is LOADED and contains no failed jobs


### Exporting Schema

In [ ]:
!dlt pipeline hn_local schema --format dbml


Found pipeline hn_local in /var/dlt/pipelines
Found schema with name hn_local
Table "stories" {
    "by" text
    "descendants" bigint
    "id" bigint [pk, not null]
    "score" bigint
    "time" bigint
    "title" text
    "type" text
    "url" text
    "_dlt_load_id" text [not null]
    "_dlt_id" text [unique, not null]
    "text" text
}

Table "stories__kids" {
    "value" bigint
    "_dlt_root_id" text [not null]
    "_dlt_parent_id" text [not null]
    "_dlt_list_idx" bigint [not null]
    "_dlt_id" text [unique, not null]
}

Table "_dlt_version" {
    "version" bigint [not null]
    "engine_version" bigint [not null]
    "inserted_at" timestamp [not null]
    "schema_name" text [not null]
    "version_hash" text [not null]
    "schema" text [not null]
    Note {
        'Created by DLT. Tracks schema updates'
    }
}

Table "_dlt_loads" {
    "load_id" text [not null]
    "schema_name" text
    "status" bigint [not null]
    "inserted_at" timestamp [not null]
    "schema_version_

You can take the output and paste it into third-party apps like dbdiagram, as shown below.


In [ ]:
from IPython.display import HTML

HTML("""
<iframe width="560" height="315" src="https://dbdiagram.io/e/693a47fde877c6307461d4be/693a4b26e877c630746225f6"></iframe>
""")


#### To verify the data further, we’ll use the [workspace dashboard.](https://dlthub.com/docs/general-usage/dashboard)

In [ ]:
import subprocess, time
from google.colab import output

# Run server in background
# The correction is here: use subprocess.Popen
subprocess.Popen(['dlt', 'pipeline', 'hn_local', 'show'])

# Increase the wait time to 10-15 seconds
print("Waiting 10 seconds for dlt server to start...")
time.sleep(10)

# Display the iframe with increased size
output.serve_kernel_port_as_iframe(
    2718,
    height=900,  # Increased Height for better visibility
    width='100%' # Set Width
)

Waiting 10 seconds for dlt server to start...


<IPython.core.display.Javascript object>

## Switch to MotherDuck
To switch to MotherDuck, we only needed to change one setting: update the destination parameter to 'motherduck.'

> **Note:**  
> Make sure your credentials are stored as environment variables in Colab Secrets.  
> You can read here how to fetch them.



Fetch the credentials from Colab Secrets and load them as environment variables so dlt can read them automatically.

In [ ]:
import os
from google.colab import userdata

c = {"DESTINATION__MOTHERDUCK__CREDENTIALS__DATABASE",
     "DESTINATION__MOTHERDUCK__CREDENTIALS__PASSWORD"}
m = [i for i in c if userdata.get(i) is None]

if m:
    print("Missing in Colab secrets:", m)
else:
    for i in c:
        os.environ[i] = userdata.get(i)
    print("MotherDuck secrets loaded into env")

MotherDuck secrets loaded into env


### Run the MotherDuck pipeline

In [ ]:
pipeline_md = dlt.pipeline(
    pipeline_name="hacker_news_pipeline_md",
    destination="motherduck",
    dataset_name="hacker_news_data",
    #dev_mode=True,
)

load_info = pipeline_md.run(hacker_news(50))
print(load_info)


Pipeline hacker_news_pipeline_md load step completed in 4.07 seconds
1 load package(s) were loaded to destination motherduck and into dataset hacker_news_data
The motherduck destination used md://motherduck:***@/dlthub_db location to store data
Load package 1765441760.230457 is LOADED and contains no failed jobs


### Confirm the Cloud load

In MotherDuck, let's confirm that the schema and data match what we validated locally.

In [ ]:
import subprocess, time
from google.colab import output

# Run server in background
# The correction is here: use subprocess.Popen
subprocess.Popen(['dlt', 'pipeline', 'hacker_news_pipeline_md', 'show'])

# Increase the wait time to 10-15 seconds
print("Waiting 10 seconds for dlt server to start...")
time.sleep(10)

# Display the iframe with increased size
output.serve_kernel_port_as_iframe(
    2718,
    height=900,  # Increased Height for better visibility
    width='100%' # Set Width
)

Waiting 10 seconds for dlt server to start...


<IPython.core.display.Javascript object>

## **Why this works**

- **Fast Feedback Loop:**  Develop and test locally without infrastructure blockers (e.g., waiting on IAM roles or environment deployments). You get answers fast.
- **What works locally works in production:** DuckLake and MotherDuck both use the same DuckDB engine, so the pipeline behaves the same in both places. Fewer surprises when you move to production.
- **Focus on logic, not config:** You get to spend more time on the extraction, transformation, and shaping how data lands at the destination, not on babysitting environments or rewriting things for production.

You write once, validate once, and trust it.

Back to the blog: [Read it here.](https://dlthub.com/blog/ducklake-to-motherduck-with-dlt)
